##### Import the Libraries

In [1]:
import pickle 
import pandas as pd
from tensorflow.keras.models import load_model

##### Load Trained Model

In [2]:
model = load_model("../models/ann_model.keras")

##### Load Model Artifacts

In [3]:
with open("../artifacts/label_encoder.pkl", "rb") as file:
    label_encoder = pickle.load(file)

with open("../artifacts/onehot_encoder.pkl", "rb") as file:
    onehot_encoder = pickle.load(file)

with open("../artifacts/standard_scaler.pkl", "rb") as file:
    standard_scaler = pickle.load(file)

##### Example Input Data

In [4]:
# Example Input Data
input_data = {
    'CreditScore': 600,
    'Geography': 'France',
    'Gender': 'Male',
    'Age': 40,
    'Tenure': 3,
    'Balance': 60000,
    'NumOfProducts': 2,
    'HasCrCard': 1,
    'IsActiveMember': 1,
    'EstimatedSalary': 50000
}

##### Preprocess Example Input Data

In [5]:
input_df = pd.DataFrame([input_data])
input_df

,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary
0,600,France,Male,40,3,60000,2,1,1,50000


In [6]:
# LabelEncode Gender Column
input_df['Gender'] = label_encoder.transform(input_df['Gender'])
input_df

,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary
0,600,France,1,40,3,60000,2,1,1,50000


In [7]:
# OneHotEncoder Geography Column
input_df = pd.DataFrame(onehot_encoder.transform(input_df), columns=onehot_encoder.get_feature_names_out())
input_df

,encode__Geography_France,encode__Geography_Germany,encode__Geography_Spain,remainder__CreditScore,remainder__Gender,remainder__Age,remainder__Tenure,remainder__Balance,remainder__NumOfProducts,remainder__HasCrCard,remainder__IsActiveMember,remainder__EstimatedSalary
0,1.0,0.0,0.0,600.0,1.0,40.0,3.0,60000.0,2.0,1.0,1.0,50000.0


In [8]:
# Refactor Column Names
input_df.columns = [col.split("__")[-1] for col in input_df.columns]
input_df

,Geography_France,Geography_Germany,Geography_Spain,CreditScore,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary
0,1.0,0.0,0.0,600.0,1.0,40.0,3.0,60000.0,2.0,1.0,1.0,50000.0


In [9]:
# Scale the Input Data
input_df.iloc[:, 3:] = standard_scaler.transform(input_df.iloc[:, 3:])
input_df

,Geography_France,Geography_Germany,Geography_Spain,CreditScore,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary
0,1.0,0.0,0.0,-0.525203,0.911867,0.109332,-0.685587,-0.27065,0.813713,0.645314,0.962936,-0.873179


##### Model Prediction

In [10]:
predicted_output = model.predict(input_df)
predicted_output

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 51ms/step


array([[0.05870469]], dtype=float32)

In [11]:
# Churn Prediction
prediction_prob = predicted_output[0][0]
print(f"Churn Probability: {prediction_prob:.2f}")
if prediction_prob > 0.5:
    print("The customer is likely to churn.")
else:
    print("The customer is not likely to churn.")

Churn Probability: 0.06
The customer is not likely to churn.
